In [47]:
#imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow

In [48]:
#data
transactions = pd.read_csv("../data/transactions_2016_2017.csv")
train = pd.read_csv("../data/customer_clv_train.csv")
test = pd.read_csv("../data/customer_clv_test.csv")

/var/folders/3m/cw1m5lkx3ml5zf_qkrwvpxph0000gn/T/ipykernel_83549/2271748159.py:2: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv("../data/transactions_2016_2017.csv")


In [49]:
transactions["order_date"] = pd.to_datetime(transactions["order_date"])
transactions["pack_date"] = pd.to_datetime(transactions["pack_date"])

# RFM Features

In [50]:
reference_date = transactions["order_date"].max()

In [51]:
# identify returns and purchases
transactions["is_return"] = transactions["sale_revenue"] < 0
transactions["is_purchase"] = transactions["sale_revenue"] > 0

In [52]:
# create customer features
customer_features = (
    transactions
    .groupby("cust_id")
    .agg(
        # Monetary
        total_revenue=("sale_revenue", "sum"),
        avg_item_revenue=("sale_revenue", "mean"),
        max_item_revenue=("sale_revenue", "max"),

        # Frequency
        n_orders=("sale_id", "nunique"),
        n_items=("sale_id", "count"),

        # Discounts
        total_discount=("sale_discount_applied", "sum"),

        # Returns
        n_returns=("is_return", "sum"),

        # Product diversity
        n_products=("prod_id", "nunique"),
        n_brands=("prod_brand", "nunique"),
        n_colors=("prod_color", "nunique"),
        n_categories=("prod_type_3", "nunique"),

        # Size behaviour
        avg_size=("prod_size", lambda x: pd.to_numeric(x, errors="coerce").mean()),
        size_std=("prod_size", lambda x: pd.to_numeric(x, errors="coerce").std()),
        n_sizes=("prod_size", "nunique"),

        # Dates
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max")
    )
    .reset_index()
)

In [53]:
customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.days

customer_features["customer_age_days"] = (
    customer_features["last_purchase"] - customer_features["first_purchase"]
).dt.days

customer_features = customer_features.drop(
    columns=["first_purchase", "last_purchase"]
)

In [54]:
customer_features["items_per_order"] = (
    customer_features["n_items"] / customer_features["n_orders"]
)

customer_features["return_rate"] = (
    customer_features["n_returns"] / customer_features["n_items"]
)

customer_features["avg_order_value"] = (
    customer_features["total_revenue"] / customer_features["n_orders"]
)

In [55]:
customer_features["size_std"] = customer_features["size_std"].fillna(0)

In [56]:
# intensity features
customer_features["orders_per_day"] = (
    customer_features["n_orders"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["items_per_day"] = (
    customer_features["n_items"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["revenue_per_day"] = (
    customer_features["total_revenue"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["frequency_recency_ratio"] = (
    customer_features["n_orders"] /
    (customer_features["recency_days"] + 1)
)

customer_features["revenue_recency_ratio"] = (
    customer_features["total_revenue"] /
    (customer_features["recency_days"] + 1)
)

customer_features["discount_per_item"] = (
    customer_features["total_discount"] /
    (customer_features["n_items"] + 1)
)

customer_features["discount_per_order"] = (
    customer_features["total_discount"] /
    (customer_features["n_orders"] + 1)
)

In [57]:
# Average Gaps

transactions_sorted = transactions.sort_values(["cust_id", "order_date"]).copy()

transactions_sorted["prev_order_date"] = (
    transactions_sorted.groupby("cust_id")["order_date"].shift()
)

transactions_sorted["days_between_orders"] = (
    transactions_sorted["order_date"] - transactions_sorted["prev_order_date"]
).dt.days

avg_gap = transactions_sorted.groupby("cust_id")["days_between_orders"].mean()
max_gap = transactions_sorted.groupby("cust_id")["days_between_orders"].max()

customer_features["avg_days_between_orders"] = customer_features["cust_id"].map(avg_gap)
customer_features["max_days_between_orders"] = customer_features["cust_id"].map(max_gap)

customer_features["avg_days_between_orders"] = (
    customer_features["avg_days_between_orders"].fillna(customer_features["recency_days"])
)
customer_features["max_days_between_orders"] = (
    customer_features["max_days_between_orders"].fillna(customer_features["recency_days"])
)

In [58]:
customer_features["ever_returned"] = (customer_features["n_returns"] > 0).astype(int)

customer_features["returns_per_order"] = (
    customer_features["n_returns"] /
    (customer_features["n_orders"] + 1)
)

# Features added by me

In [59]:
# add delivery_days to transactions for avg_delivery_days feature
transactions["delivery_days"] = (transactions["pack_date"] - transactions["order_date"]).dt.days

# aggregate new features at customer level
new_features = (
    transactions
    .groupby("cust_id")
    .agg(
        web_only_rate=("prod_web_only", "mean"),
        outlet_rate=("prod_outlet", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        n_seasons=("prod_season", "nunique"),
    )
    .reset_index()
)

# merge into customer_features
customer_features = customer_features.merge(new_features, on="cust_id", how="left")


In [60]:
# Spending trend features
# find midpoint of the observation period
midpoint_date = transactions["order_date"].min() + (
    transactions["order_date"].max() - transactions["order_date"].min()
) / 2

# Split transactions into first half and second half based on the midpoint date
transactions['is_second_half'] = transactions['order_date'] >= midpoint_date

# aggregate per customer 
first_half_revenue = (transactions[transactions['is_second_half'] == False].groupby('cust_id')['sale_revenue'].sum())
second_half_revenue = (transactions[transactions['is_second_half'] == True].groupby('cust_id')['sale_revenue'].sum())

# Spending trend: positive if spending increased in the second half, negative if it decreased
customer_features['spending_trend'] = customer_features['cust_id'].map(second_half_revenue - first_half_revenue)

# filling Nans with 0 (customers with no transactions in one half will have NaN, but we can interpret that as no change in spending)
customer_features["spending_trend"] = customer_features["spending_trend"].fillna(0)

In [61]:
customer_features.to_csv("../data/customer_features.csv", index=False)